<div style="background-color:#fff4e6; border-left:8px solid #cc7a00; padding:20px; margin:20px 0;">
  <h2 style="color:#994d00;"><strong>Disclaimer</strong></h2>
  <p style="color:#333333;">
    This project uses data sourced from the <strong>ImgFlip575K</strong> dataset, which contains text extracted from memes available online.
    The content in this dataset may include opinions, biases, offensive language, or politically sensitive statements.
    These texts do <strong>not</strong> represent the views or beliefs of the authors of this project.
  </p>
  <p style="color:#333333;">
    All use of the data is strictly for academic and research purposes, and care has been taken to apply preprocessing steps
    (e.g., cleaning, formatting, anonymization, zero-shot emotion labeling) to mitigate the impact of any potentially harmful content.
  </p>
</div>

# NLP Model Setup: ImgFlip Dataset Strategy

## Dataset Switching Configuration

**This notebook follows the unified dataset strategy from 2a!** We can easily switch between:

- **GoEmotions**: Pre-labeled emotion dataset from Google Research
- **ImgFlip575K**: Meme captions with zero-shot emotion labeling (current focus)

### How to Switch Datasets

1. **Change `DATASET_NAME`** in the configuration cell below
2. **Run the notebook normally** - the unified factory handles everything else!

## Objective

This notebook demonstrates **unified controlled text generation** using fine-tuned language modeling. We generate emotionally aligned captions conditioned on user-provided mood labels (e.g., joy, sadness, anger) using **ImgFlip dataset**.

**Task Type:** Conditional Text Generation with Emotional Alignment

**Models:** GPT-2 fine-tuned on ImgFlip575K dataset with zero-shot emotion labels

**Final Use Case:** Creative captioning over cartoonized images for personalized meme generation

**Key Innovation:** Unified dataset pipeline that handles both pre-labeled and unlabeled data transparently, enabling fair comparison and easy experimentation.

### Unified Dataset Strategy Details

This notebook implements our **Unified Dataset Factory Pattern** that solves the core challenge: *"ImgFlip is missing the label right?"*

| Dataset | Label Status | Processing Method |
|---------|-------------|------------------|
| **GoEmotions** |  **Pre-labeled** | Direct mapping: `labels` → `mood` |
| **ImgFlip575K** |  **Unlabeled** | Zero-shot emotion classification with `facebook/bart-large-mnli` |

**Key Benefits:**
- **Same Pipeline**: Identical processing chain regardless of dataset choice
- **Easy Switching**: Change `DATASET_NAME` parameter to switch between datasets  
- **Automatic Labeling**: Zero-shot emotion classification applied transparently for unlabeled data
- **Consistent Output**: All datasets result in the same `mood` + `caption` format
- **Fair Comparison**: Same evaluation metrics and training procedures


## Literature Review & Method Selection Rationale

### Why GPT-2 for Conditional Text Generation?

We chose GPT-2 for its strong generative performance and ease of conditioning via prompt engineering. GPT-2 has been widely used for custom text generation tasks and shows excellent results when fine-tuned on domain-specific data.

**Key Research Supporting Our Approach:**

1. **Woolf (2019)** – ["How To Make Custom AI-Generated Text With GPT-2"](https://minimaxir.com/2019/09/howto-gpt2/) demonstrates that GPT-2 can be effectively fine-tuned for specific text styles and domains using relatively small datasets.

2. **Meme Captioning Research** – Studies like [XMeCap (Wang et al., 2024)](https://arxiv.org/abs/2407.17152) and [MemeCap (Sharma et al., 2023)](https://aclanthology.org/2023.emnlp-main.89/) show that controlled text generation for visual content requires emotional alignment and contextual understanding.

3. **Controlled Generation** – Recent work in controllable text generation, such as [Plug and Play Language Models (Dathathri et al., 2020)](https://arxiv.org/abs/1912.02164), shows that prompt-based conditioning enables emotional alignment with minimal supervision.

### Alternative Models Considered:

- **GPT-Neo**: Larger model but slower inference, limited practical difference for our use case
- **BERT**: Not generative, unsuitable for caption generation
- **T5**: Text-to-text approach could work but requires more complex conditioning setup
- **Custom LSTM/GRU**: Would require training from scratch, insufficient capacity for creative generation

### Dataset Choice: ImgFlip575K

The ImgFlip575K dataset provides high-quality meme captions with emotional diversity, making it ideal for training mood-conditioned generation models. Its meme-specific content aligns perfectly with our goal of generating contextually appropriate captions for cartoonized images.

In [1]:
!nvidia-smi

Sun Aug 10 15:02:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

CUDA available: True
Device: cuda


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Install required packages from requirements.txt:
import sys
!{sys.executable} -m pip install -r "/content/drive/MyDrive/northeastern/ie7374/PerToon/requirements.txt"

In [5]:
import pandas as pd
import requests, json
import torch
import os
import matplotlib.pyplot as plt
import seaborn as sns
import math
from datasets import load_dataset, Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling, pipeline
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

In [6]:
# Project root configuration
PROJECT_ROOT = "/content/drive/MyDrive/northeastern/ie7374/PerToon"

In [7]:
# DATASET CONFIGURATION - CHANGE HERE TO SWITCH DATASETS!
DATASET_NAME = "imgflip"  # Options: "goemotions" or "imgflip"

print(f"Selected Dataset: {DATASET_NAME.upper()}")
print("Dataset overrides will be configured in the next cell with factory setup.")

Selected Dataset: IMGFLIP
Dataset overrides will be configured in the next cell with factory setup.


In [8]:
# =============================================================================
# UNIFIED DATASET CONFIGURATION & LOADING
# =============================================================================

# Define complete DATASET_OVERRIDES for this specific experiment
DATASET_OVERRIDES = {
    # Processing parameters
    "captions_to_process": 40000,  # Smaller sample for 2c notebook
    "batch_size": 100,

    # File paths for this specific experiment (configurable cache files)
    "raw_path": "captions_imgflip.csv",              # raw captions from GitHub
    "sampled_path": "captions_imgflip_sample.csv",   # sampled subset
    "labeled_path": "mood_captions_imgflip_sample.csv"  # labeled subset
}

print(f"Selected Dataset: {DATASET_NAME.upper()}")
print(f"Dataset Overrides: {DATASET_OVERRIDES}")

# Add helpers to path and import the factory
import sys
sys.path.append(f"{PROJECT_ROOT}/helpers")
from dataset_factory import create_dataset_factory, DATASET_CONFIGS

# Create factory instance
factory = create_dataset_factory(PROJECT_ROOT)

print(f"\nUnified factory loaded successfully!")
print(f"Available datasets: {list(DATASET_CONFIGS.keys())}")

# Show the configuration for the selected dataset
config = DATASET_CONFIGS[DATASET_NAME]
print(f"\n{DATASET_NAME.upper()} Configuration:")
print(f"   Pre-labeled: {'Yes' if config.has_labels else 'No (will apply zero-shot labeling)'}")
print(f"   Labeler model: {getattr(config, 'labeler_model', 'N/A')}")
print(f"   Raw path: {DATASET_OVERRIDES['raw_path']}")
print(f"   Sampled path: {DATASET_OVERRIDES['sampled_path']}")
print(f"   Labeled path: {DATASET_OVERRIDES['labeled_path']}")

# Load the dataset using the unified factory with our overrides
# The factory will now handle everything: caching, downloading, sampling, and labeling.
print(f"\nLoading {DATASET_NAME} dataset with unified pipeline...")
df = factory.load_dataset(DATASET_NAME, **DATASET_OVERRIDES)

print(f"\nDataset ready for EDA and Training!")
print(f"Shape: {df.shape}")
print(f"Unique emotions: {df['mood'].nunique()}")
print(f"Column names: {list(df.columns)}")
df.head()

Selected Dataset: IMGFLIP
Dataset Overrides: {'captions_to_process': 40000, 'batch_size': 100, 'raw_path': 'captions_imgflip.csv', 'sampled_path': 'captions_imgflip_sample.csv', 'labeled_path': 'mood_captions_imgflip_sample.csv'}

Unified factory loaded successfully!
Available datasets: ['goemotions', 'imgflip']

IMGFLIP Configuration:
   Pre-labeled: No (will apply zero-shot labeling)
   Labeler model: facebook/bart-large-mnli
   Raw path: captions_imgflip.csv
   Sampled path: captions_imgflip_sample.csv
   Labeled path: mood_captions_imgflip_sample.csv

Loading imgflip dataset with unified pipeline...
Loading imgflip dataset...
    Found pre-labeled data. Loading from: /content/drive/MyDrive/northeastern/ie7374/PerToon/data/mood_captions_imgflip_sample.csv

Dataset ready for EDA and Training!
Shape: (200000, 2)
Unique emotions: 28
Column names: ['mood', 'caption']


,mood,caption
0,confusion,"DAY 20 OF QUARANTINE ""WHAT'S A TREE?"""
1,disapproval,DRANK 19 CORONAS WONT RISK SPREADING CORONA BI...
2,confusion,HOW ARE WE SUPPOSED TO SNEEZE AND COUGH INTO O...
3,disappointment,"THERE SHOULD BE WATERMELON, FIREMELON, EARTHME..."
4,confusion,"If you rotate the word ""pod"" it still spells ""..."


### Pre-labeled Only Step

The zero-shot labeling above is applied exclusively to the sampled subset stored at `data/mood_captions_imgflip_sample.csv`. If this CSV exists, the notebook skips both download and labeling, reusing the cached results. This mirrors 2a's practice of working with a bounded subset (40k words).


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Exploratory Data Analysis (EDA)</strong></h2>
  <p style="color:#333333;">Before proceeding with model training, we conduct comprehensive EDA to understand our dataset characteristics, identify potential challenges, and inform our preprocessing decisions.</p>
</div>

## Methodology: Data-Driven Experimental Design

### Why EDA First?

Before establishing baselines or fine-tuning, we begin with exploratory data analysis (EDA) to understand the dataset's structure, emotional distribution, and text properties.

This evidence-driven approach ensures that:
- Our baseline testing uses emotions with meaningful representation.
- We identify class imbalance risks early.
- Preprocessing and tokenization are tailored to actual data characteristics.



In [13]:
# Execute data preparation on the cached, labeled sample (avoiding full dataset work)
print("Data Preparation on Cached Labeled Sample Complete!")
print("=" * 50)

print(f"Using sampled+labeled dataset:")
print(f"Shape: {df.shape}")
print(f"Unique emotions: {df['mood'].nunique()}")

print("\n" + "=" * 50)
# Perform comprehensive EDA using factory method
emotion_counts, caption_lengths, word_counts = factory.perform_eda_analysis(df, "ImgFlip (sample)")

Data Preparation on Cached Labeled Sample Complete!
Using sampled+labeled dataset:
Shape: (200000, 2)
Unique emotions: 28


 EXPLORATORY DATA ANALYSIS - IMGFLIP (SAMPLE)
1. DATASET OVERVIEW
   Total samples: 200,000
   Number of unique emotions: 28
   Average caption length: 69.5 characters
   Median caption length: 60.0 characters

2. TEXT LENGTH STATISTICS
   Min length: 1 characters
   Max length: 3990 characters
   25th percentile: 41.0 characters
   75th percentile: 88.0 characters
   Standard deviation: 44.9 characters

3. WORD COUNT STATISTICS
   Average words per caption: 13.2
   Median words per caption: 12.0
   Min words: 1
   Max words: 715

4. COMPLETE EMOTION DISTRIBUTION
   Emotion (Count | Percentage)
   -----------------------------------
   disapproval  (52481 |  26.2%)
   confusion    (39558 |  19.8%)
   surprise     (32635 |  16.3%)
   disappointment (11606 |   5.8%)
   annoyance    (10190 |   5.1%)
   desire       ( 6856 |   3.4%)
   caring       ( 6668 |   3.3%)
  

## EDA Insights & Implications for Model Training

Our exploratory data analysis provided several insights that directly inform model training strategies:

### 1. Text Length Diversity
The dataset contains captions with a wide range of lengths. To accommodate this variability, we set a maximum tokenization length of `128`, which effectively captures the majority of examples while truncating outliers.

### 2. Class Imbalance Challenge
EDA revealed a **severe class imbalance**:
- The most represented emotion has many samples, making up a large percentage of the dataset.
- The least represented emotions have very few samples.
- The resulting **imbalance ratio** requires careful attention.

### 3. High Caption Uniqueness
Over **95%** of the captions are unique, indicating high linguistic diversity. This reduces overfitting risk and supports rich language modeling.

### 4. Strong Data Quality
The dataset has:
- No missing captions
- Very few extremely short or long examples

This ensures the model receives clean, well-formed inputs without the need for heavy preprocessing.

### 5. Implications for Training
Given the above findings:
- **Balancing the dataset** is essential to ensure fair learning across all emotion classes.
- The data is suitable for training a GPT-2 model with minimal cleaning.
- **Fine-tuning strategies** should monitor performance on underrepresented emotions.
- Emotion-specific evaluation metrics, such as polarity alignment, will be used to assess the effectiveness of mood-conditioned generation.

<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Addressing Dataset Class Imbalance</strong></h2>
  <p style="color:#333333;">Balancing the dataset is essential to ensure fair learning across all emotion classes.</p>
</div>

In [14]:
# UNIFIED DATASET PROCESSING

# Use unified factory methods for consistency
balanced_df = factory.balance_dataset_for_training(
    df,
    max_samples_per_emotion=1500,
    min_samples_per_emotion=100
)


 ADDRESSING CLASS IMBALANCE
   disapproval: 52481 → 1500 (downsampled)
   confusion: 39558 → 1500 (downsampled)
   surprise: 32635 → 1500 (downsampled)
   disappointment: 11606 → 1500 (downsampled)
   annoyance: 10190 → 1500 (downsampled)
   desire: 6856 → 1500 (downsampled)
   caring: 6668 → 1500 (downsampled)
   approval: 6522 → 1500 (downsampled)
   realization: 5853 → 1500 (downsampled)
   relief: 5468 → 1500 (downsampled)
   admiration: 5174 → 1500 (downsampled)
   amusement: 3339 → 1500 (downsampled)
   curiosity: 2616 → 1500 (downsampled)
   optimism: 2148 → 1500 (downsampled)
   excitement: 1360 (unchanged)
   neutral: 1138 (unchanged)
   remorse: 923 (unchanged)
   embarrassment: 832 (unchanged)
   gratitude: 711 (unchanged)
   nervousness: 661 (unchanged)
   fear: 547 (unchanged)
   love: 521 (unchanged)
   anger: 491 (unchanged)
   pride: 429 (unchanged)
   disgust: 376 (unchanged)
   sadness: 364 (unchanged)
   joy: 330 (unchanged)
   grief: 203 (unchanged)

 Balanced data

## Class Imbalance Successfully Addressed

### Balancing Strategy Applied:

Our `balance_dataset_for_training()` function implements a **hybrid sampling approach**:

1. **Downsampling**: Limit over-represented emotions to **1,500 samples max**
   - Prevents dominant emotions from overwhelming training
   - Maintains data quality by keeping diverse examples

2. **Upsampling**: Ensure under-represented emotions have **100 samples min**
   - Uses replacement sampling to boost rare emotions
   - Gives every emotion fair learning opportunity

3. **Shuffling**: Randomize the balanced dataset to prevent order bias

### Impact on Training Quality:

**Before Balancing:**
- Extreme bias toward dominant emotions
- Poor learning for rare emotions
- High risk of mode collapse and repetitive generation

**After Balancing:**
- **~15:1** maximum imbalance ratio (much more manageable)
- All emotions have meaningful representation
- Better emotion conditioning expected
- More stable training dynamics

### Expected Model Improvements:

• **Diverse Emotion Generation**: Model can now learn patterns for all 28 emotions  
• **Reduced Bias**: No single emotion dominates the training signal  
• **Better Evaluation**: Fairer performance assessment across emotion categories  
• **Stable Training**: Balanced gradients prevent training instability

## Baseline Experiments: Prepare for Zero-Shot vs Fine-Tuned Comparison

### Data-Driven Emotion Selection (Post-Balancing)

Now that we understand our training data through EDA and have applied dataset balancing, we can make informed decisions about baseline testing:

**Based on Post-Balancing Distribution:**
- **High-frequency emotions** (1,500 samples): Most common emotions after balancing
- **Medium-frequency emotions** (100-1,499 samples): Emotions with natural medium frequency
- **Low-frequency emotions** (100 samples, upsampled): Originally rare emotions that were boosted

**Baseline Strategy:**
We'll test a **representative sample** spanning the balanced frequency spectrum to understand:
1. How well GPT-2 handles emotions with maximum representation
2. Performance on naturally medium-frequency emotions that weren't resampled
3. Challenges with artificially upsampled low-frequency emotions

**Key Insight:** Post-balancing, we expect more consistent performance across emotions since the extreme imbalance has been reduced to a manageable ratio. This gives us realistic baselines for comparison after fine-tuning on a more balanced dataset.


## Understanding Sentiment Polarity for Evaluation

**Polarity** is a key metric in sentiment analysis that measures the emotional tone of text on a scale from -1 to +1:

- **Positive Values (+0.1 to +1.0)**: Indicate positive sentiment (joy, happiness, love, excitement)
- **Negative Values (-0.1 to -1.0)**: Indicate negative sentiment (sadness, anger, fear, disgust)  
- **Neutral Values (~0.0)**: Indicate neutral or objective text

**Why Polarity Matters for Our Project:**

In our mood-conditioned caption generation task, polarity serves as a quantitative measure to evaluate whether generated captions align with the intended emotional mood. For example:

- A caption generated for mood "joy" should ideally have a positive polarity (>0.1)
- A caption for mood "sadness" should have a negative polarity (<-0.1)
- A caption for mood "anger" should also have negative polarity

High-frequency emotions should show better improvement after fine-tuning, while low-frequency emotions may remain challenging even after training.

**Our Data-Driven Evaluation Strategy:**

We use TextBlob's sentiment analysis to calculate polarity scores, which helps us:
1. **Establish informed baselines** after understanding our training data through EDA
2. **Quantitatively compare** fine-tuned vs baseline models on relevant emotions
3. **Measure improvement** in emotional alignment across different frequency categories


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Baseline Experiments - Zero-Shot</strong></h2>
  <p style="color:#333333;">Setup the benchmark for model tuning performance comparison.</p>
</div>

In [15]:
# Initialize zero-shot GPT-2 pipeline for baseline testing
from transformers import set_seed

print("BASELINE POLARITY TESTING")
print("=" * 60)
print("Testing emotions selected based on post-balancing distribution...")

# Reproducible sampling
set_seed(42)

zero_shot_generator = pipeline("text-generation", model="gpt2", tokenizer="gpt2")

# Post-balancing emotion selection spanning balanced frequency spectrum
test_moods = ['joy', 'sadness', 'anger', 'love', 'fear', 'neutral', 'surprise', 'excitement']

print("\nBASELINE RESULTS (Zero-Shot GPT-2) - Post-Balancing Selection:")
print("=" * 70)

all_results = {}
for mood in test_moods:
    # Use more natural prompts that GPT-2 can better understand
    if mood == "neutral":
        prompt = "Caption: "
    else:
        prompt = f"I feel {mood}. Caption: "

    # Generate caption
    output = zero_shot_generator(prompt, max_new_tokens=40, num_return_sequences=1,
                                do_sample=True, temperature=0.8, top_p=0.9,
                                pad_token_id=50256, eos_token_id=50256)
    generated_text = output[0]["generated_text"].replace(prompt, "").strip()

    # Calculate sentiment polarity using TextBlob
    polarity = TextBlob(generated_text).sentiment.polarity
    all_results[mood] = {"text": generated_text, "polarity": polarity}

    print(f"Mood: {mood}")
    print(f"Generated: {generated_text}")
    print(f"Polarity: {polarity:.3f}")
    print("-" * 30)

overall_avg = sum(r["polarity"] for r in all_results.values()) / len(all_results)
print(f"\nOverall Baseline Average Polarity: {overall_avg:.3f}")
print("Baseline established with balanced dataset emotion selection.")

BASELINE POLARITY TESTING
Testing emotions selected based on post-balancing distribution...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0



BASELINE RESULTS (Zero-Shot GPT-2) - Post-Balancing Selection:
Mood: joy
Generated: A friend of mine told me, "I have a friend who was diagnosed with schizophrenia at the time." The friend said, "He came to see me." That was my friend. He said
Polarity: 0.000
------------------------------
Mood: sadness
Generated: We have a lot of questions.
I have some thoughts about how to deal with this. I think that when we come together, we can do things. I think that when we don't
Polarity: 0.000
------------------------------
Mood: anger
Generated: My son. I feel anger.
In a way, my mother's decision to take a step back and take her son back was a direct reaction to her own feelings about the situation and not
Polarity: 0.000
------------------------------
Mood: love
Generated:  President Trump shakes hands with a former top CIA official on Capitol Hill in Washington on March 14. Hide Caption 1 of 5 Photos: President Trump's first 100 days In the Oval Office,
Polarity: 0.250
-------------------

## Intelligent Data Cleaning, Text Processing & Complete Output Strategy

### Problem with Early Splitting Approaches
Traditional meme generation approaches often break text prematurely:
- Brittle splitting rules that break complete thoughts
- Length truncation causing incomplete ideas
- Lost context when separating related concepts

### Our Improved Solution: Complete Output Generation
The `create_complete_meme_training_dataset()` function implements intelligent processing by:
1. Smart text cleaning (preserves sentence structure and apostrophes)
2. Complete thought preservation (no premature cuts)
3. Natural length limits
4. Context-aware processing to maintain emotional flow
5. Flexible training format: prompts like `Generate a {mood} meme caption: {complete_thought}<|endoftext|>`

### Key Innovation: Post-Processing Intelligence
- Train on complete outputs; format later with `intelligent_meme_parser()`
- Split at logical points (sentences/connectors/commas)
- Maintain emotional integrity by keeping related concepts together

### Expected Benefits
- Coherent captions
- Better emotion alignment
- Flexible formatting
- Production-ready outputs
- Robust to short and long texts


### Dataset sizes used for fair comparison

- GoEmotions (notebook 2a) — balanced_df size: ≈ 23,030 (from 36,308 single‑label samples), with max 1,500 and min 100 per emotion.
- ImgFlip (this notebook 2c) — balanced_df size: ≈ 29,886 (from 200,000 labeled samples), with the same max 1,500 and min 100 per emotion.

We keep these sizes as-is for experiments. The small difference is acceptable; both datasets use identical balancing parameters, so comparisons remain fair. No additional sampling is applied.


In [ ]:
# Verify cleaning is working by showing before/after examples
print("\nMeme formatting examples:")
if 'df' in locals() and len(df) > 0:
    import random
    sample_indices = random.sample(range(len(df)), min(10, len(df)))
    for i, idx in enumerate(sample_indices):
        sample_original = df['caption'].iloc[idx]
        sample_cleaned = factory._clean_text_for_memes(sample_original, max_words=15)
        if sample_original != sample_cleaned or len(sample_original) > 50 or any(char in sample_original for char in ['@', '#', 'http', '>', '<']):
            print(f"  Original caption: '{sample_original[:60]}{'...' if len(sample_original) > 60 else ''}'")
            print(f"  Cleaned caption:  '{sample_cleaned.upper()}'")
            print()
        elif i < 3:
            print(f"  Original caption: '{sample_original[:60]}{'...' if len(sample_original) > 60 else ''}'")
            print(f"  Cleaned caption:  '{sample_cleaned.upper()}'")
            print()
else:
    print("Dataset not available for verification")

print("Text cleaning step is active in the unified pipeline")
print("Applied before tokenization in factory.create_training_dataset()")


In [ ]:
# Build unified training dataset ONCE, then tokenize
training_dataset = factory.create_complete_meme_training_dataset(
    balanced_df,
    max_words=15
)

tokenized_dataset, tokenizer = factory.tokenize_dataset(training_dataset)

print(f"\nUnified training dataset prepared!")
print(f"   Examples: {len(tokenized_dataset)}")
print(f"   Using complete output approach (preserves context)")
print(f"   Text cleaning: Updated to preserve apostrophes (e.g., can't vs cant)")


In [ ]:
# Visualization and training summary (mirroring 2a)

def plot_training_metrics(model_configs, trained_models):
    import pandas as pd
    import matplotlib.pyplot as plt
    if not trained_models:
        print("No models were successfully trained. Skipping visualization.")
        return
    print("Creating training metrics visualization from actual logs...")
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle('Training Metrics Comparison: GPT-2 Base vs Medium', fontsize=16, fontweight='bold')
    colors = {'base': '#1f77b4', 'medium': '#ff7f0e'}
    for size, model_info in trained_models.items():
        if 'trainer' not in model_info:
            continue
        log_history = model_info['trainer'].state.log_history
        if not log_history:
            continue
        logs_df = pd.DataFrame(log_history)
        train_logs = logs_df[logs_df['loss'].notna()].copy()
        if not train_logs.empty:
            axes[0].plot(train_logs['step'], train_logs['loss'], label=f'{size.title()} Model', color=colors.get(size, '#555'), linewidth=2, alpha=0.8)
        eval_logs = logs_df[logs_df['eval_loss'].notna()].copy()
        if not eval_logs.empty:
            axes[1].plot(eval_logs['step'], eval_logs['eval_loss'], label=f'{size.title()} Model', marker='o', linestyle='--', color=colors.get(size, '#555'), linewidth=2)
        lr_logs = logs_df[logs_df['learning_rate'].notna()].copy()
        lr_logs = lr_logs[lr_logs['step'] > 200]
        if not lr_logs.empty:
            axes[2].plot(lr_logs['step'], lr_logs['learning_rate'], label=f'{size.title()} Model', color=colors.get(size, '#555'), linewidth=2)
    axes[0].set_title('Training Loss Over Time'); axes[0].set_xlabel('Training Steps'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].set_title('Validation Loss Over Time'); axes[1].set_xlabel('Training Steps'); axes[1].set_ylabel('Validation Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].set_title('Learning Rate Schedule'); axes[2].set_xlabel('Training Steps'); axes[2].set_ylabel('Learning Rate'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    import os
    figures_dir = f"{globals().get('PROJECT_ROOT', '..')}/figures"
    os.makedirs(figures_dir, exist_ok=True)
    plot_path = f"{figures_dir}/training_metrics_comparison_actual.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Training metrics visualization saved to {plot_path}")


def create_training_summary_table(trained_models):
    import pandas as pd
    import math
    print("TRAINING RESULTS SUMMARY")
    print("=" * 80)
    results = []
    for size, model_info in trained_models.items():
        params = "117M" if size == 'base' else "345M"
        final_loss_str = "N/A"; perplexity_str = "N/A"; status = "Incomplete"
        if 'trainer' in model_info and hasattr(model_info['trainer'], 'state'):
            trainer = model_info['trainer']
            log_history = trainer.state.log_history
            if log_history:
                train_loss_logs = [log.get('loss') for log in log_history if 'loss' in log]
                if train_loss_logs:
                    final_loss_str = f"{train_loss_logs[-1]:.4f}"; status = "Completed"
                eval_loss_logs = [log.get('eval_loss') for log in log_history if 'eval_loss' in log]
                if eval_loss_logs:
                    best_eval_loss = min(eval_loss_logs)
                    perplexity = math.exp(best_eval_loss)
                    perplexity_str = f"{perplexity:.4f}"
        results.append({
            "Model Description": model_info.get('description', size.title()),
            "Parameters": params,
            "Final Training Loss": final_loss_str,
            "Best Perplexity": perplexity_str,
            "Status": status
        })
    summary_df = pd.DataFrame(results)
    print(summary_df.to_string(index=False))
    print("=" * 80)

# If training ran, show summary and metrics
if 'trained_models' in locals() and trained_models:
    create_training_summary_table(trained_models)
    plot_training_metrics(model_configs, trained_models)
else:
    print("No trained models found for visualization. Run the training cells first.")


## Model Fine-Tuning

Train GPT-2 on the balanced, meme-formatted dataset using the unified factory approach. Checkpoints are auto-detected for resume.


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Model Fine-Tuning</strong></h2>
  <p style="color:#333333;">Train GPT-2 on our balanced, meme-formatted dataset to generate emotionally aligned meme-style captions. This section implements the fine-tuning pipeline using the unified factory approach from 2a notebook.</p>
</div>

In [16]:
# CLEAR MODEL NAMING STRATEGY
print("Target Models (5 total):")
print("1. gpt2-base-original     - Base model (no training)")
print("2. gpt2-base-goemotions   - Base + GoEmotions dataset")
print("3. gpt2-medium-goemotions - Medium + GoEmotions dataset")
print("4. gpt2-base-imgflip      - Base + ImgFlip dataset")
print("5. gpt2-medium-imgflip    - Medium + ImgFlip dataset")
print()

# Current dataset being processed
print(f"Currently processing: {DATASET_NAME.upper()} dataset")
print()

# Model configurations with CLEAR naming
if DATASET_NAME == "imgflip":
    model_configs = [
        {
            "size": "base",
            "output_dir": f"{PROJECT_ROOT}/models/gpt2-base-imgflip",
            "description": "GPT-2 Base + ImgFlip (117M params)"
        },
        {
            "size": "medium",
            "output_dir": f"{PROJECT_ROOT}/models/gpt2-medium-imgflip",
            "description": "GPT-2 Medium + ImgFlip (345M params)"
        }
    ]
elif DATASET_NAME == "goemotions":
    model_configs = [
        {
            "size": "base",
            "output_dir": f"{PROJECT_ROOT}/models/gpt2-base-goemotions",
            "description": "GPT-2 Base + GoEmotions (117M params)"
        },
        {
            "size": "medium",
            "output_dir": f"{PROJECT_ROOT}/models/gpt2-medium-goemotions",
            "description": "GPT-2 Medium + GoEmotions (345M params)"
        }
    ]

print("Models for this run:")
for config in model_configs:
    print(f"   {config['description']}: {config['output_dir']}")

Target Models (5 total):
1. gpt2-base-original     - Base model (no training)
2. gpt2-base-goemotions   - Base + GoEmotions dataset
3. gpt2-medium-goemotions - Medium + GoEmotions dataset
4. gpt2-base-imgflip      - Base + ImgFlip dataset
5. gpt2-medium-imgflip    - Medium + ImgFlip dataset

Currently processing: IMGFLIP dataset

Models for this run:
   GPT-2 Base + ImgFlip (117M params): /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip
   GPT-2 Medium + ImgFlip (345M params): /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-medium-imgflip


In [ ]:
print("=" * 80)
print(f"TRAINING BOTH BASE AND MEDIUM MODELS ON {DATASET_NAME.upper()} DATASET")
print("=" * 80)

# Use the model_configs from the previous cell (which have clear naming)
print(f"Will train {len(model_configs)} models for {DATASET_NAME} dataset:")
for config in model_configs:
    print(f"   {config['description']}")
print()

# Import enhanced training function from helpers
from caption_helpers import fine_tune_gpt2_enhanced, get_checkpoint_or_none

trained_models = {}

for config in model_configs:
    print(f"\n{'='*60}")
    print(f"TRAINING: {config['description']}")
    print(f"{'='*60}")

    # Check for existing checkpoints
    resume_checkpoint = get_checkpoint_or_none(config['output_dir'])

    # Clear GPU memory before training each model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        # Dataset has already been prepared and tokenized globally
        trainer = fine_tune_gpt2_enhanced(
            tokenized_dataset=tokenized_dataset,
            tokenizer=tokenizer,
            output_dir=config['output_dir'],  # Use proper dataset-specific paths
            epochs=3,  # Reasonable epochs for testing
            batch_size=4,  # Will be adjusted automatically for medium model
            learning_rate=2e-5,
            resume_from_checkpoint=resume_checkpoint,
            model_size=config['size']  # Now properly supports base/medium
        )

        trained_models[config['size']] = {
            'trainer': trainer,
            'output_dir': config['output_dir'],
            'description': config['description']
        }

        print(f"\n {config['description']} training completed successfully!")

    except KeyboardInterrupt:
        print(f"\n⏸ {config['description']} training was interrupted.")
        print("You can resume later using the checkpoint.")
        break

    except Exception as e:
        print(f"\n {config['description']} training failed: {e}")
        print("Check the checkpoint directory for partial progress.")
        import traceback
        traceback.print_exc()

        # Clear memory and continue with next model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n{'='*80}")
print(f"ENHANCED TRAINING SUMMARY FOR {DATASET_NAME.upper()}")
print(f"{'='*80}")
if trained_models:
    print(f" Successfully trained {len(trained_models)} models:")
    for size, info in trained_models.items():
        print(f"   {info['description']}: {info['output_dir']}")
    print()
    print(" Your model collection progress:")
    print("   Current dataset models:  Complete")
    if DATASET_NAME == "imgflip":
        print("   GoEmotions models: Use 2a notebook or set DATASET_NAME='goemotions'")
    else:
        print("   ImgFlip models: Set DATASET_NAME='imgflip' and run again")
else:
    print(" No models trained successfully")
    print("Check the error messages above and try again")
print(f"{'='*80}")

TRAINING BOTH BASE AND MEDIUM MODELS ON IMGFLIP DATASET
Will train 2 models for imgflip dataset:
   GPT-2 Base + ImgFlip (117M params)
   GPT-2 Medium + ImgFlip (345M params)


TRAINING: GPT-2 Base + ImgFlip (117M params)
No output directory found, cannot search for checkpoints.

 CREATING COMPLETE MEME TRAINING DATASET
Using complete output strategy (preserving complete thoughts)...
After intelligent processing: 29292 complete captions remain
 Created complete meme training dataset with 29292 examples
 Key features:
   - No early TOP/BOTTOM splitting
   - Preserves complete thoughts and emotional context
   - Allows for intelligent post-processing of generated output
   - Maintains sentence boundaries and logical flow
   - Better emotional alignment through preserved context

 TOKENIZING DATASET


Map:   0%|          | 0/29292 [00:00<?, ? examples/s]

 Tokenized 29292 examples
Starting GPT-2 Fine-Tuning (base model)...
Loading GPT-2 Base model (117M parameters)
Loaded gpt2 model
Created validation split: 26363 train, 2929 validation
Added early stopping callback with patience=3


Using auto half precision backend
***** Running training *****
  Num examples = 26,363
  Num Epochs = 3
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 2
  Total optimization steps = 9,885
  Number of trainable parameters = 124,439,808
The following columns in the Training set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text. If text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.


Enhanced Training Configuration:
   Dataset size: 29292 examples
   Validation split: Yes
   Epochs: 3
   Batch size: 4
   Learning rate: 2e-05
   Metrics: Loss+ Early Stopping
   Output directory: /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip
   Using GPU: True
   Resume from checkpoint: Starting fresh

Starting enhanced training with metrics monitoring...
Training will log: Step, Training Loss, Validation Loss, Learning Rate


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,2.990100,2.871673
1000,2.850800,2.811351
1500,2.766200,2.769655



***** Running Evaluation *****
  Num examples = 2929
  Batch size = 2
The following columns in the Evaluation set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: text. If text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.
Saving model checkpoint to /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip/checkpoint-500
Configuration saved in /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip/checkpoint-500/config.json
Configuration saved in /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip/checkpoint-500/generation_config.json
Model weights saved in /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip/checkpoint-500/model.safetensors
tokenizer config file saved in /content/drive/MyDrive/northeastern/ie7374/PerToon/models/gpt2-base-imgflip/checkpoint-500/tokenizer_config.json
Special tokens file saved in /content/driv

## Summary & Next Steps

### What We Accomplished

1. **Defined Clear Objectives**: Established controlled text generation as our NLP task with emotional alignment goals using the **ImgFlip dataset strategy**

2. **Literature Review**: Justified GPT-2 selection based on research in controlled generation and meme captioning, specialized for meme content

3. **Baseline Establishment**: Tested zero-shot GPT-2 performance across balanced emotion categories for comparison with fine-tuned model

4. **Unified Dataset Integration**: Successfully applied the unified factory approach from 2a notebook to handle ImgFlip dataset with zero-shot emotion labeling

5. **Enhanced Training Pipeline**: Successfully fine-tuned both Base and Medium GPT-2 models with comprehensive metrics including:
   -  **Perplexity tracking** and validation splits
   -  **Text cleaning verification** (meme-specific processing)
   -  **Complete output strategy** (preserving emotional context)
   -  **Checkpoint resume support** for interrupted training

6. **Modular Implementation**: Reused enhanced functions from unified factory for data preparation, tokenization, and model training

### Model Outputs

- **Fine-tuned Models**: `{PROJECT_ROOT}/models/gpt2-base-imgflip/` and `gpt2-medium-imgflip/`
- **Training Data**: `{PROJECT_ROOT}/data/mood_captions_imgflip.csv` (balanced dataset)
- **Unified Factory**: Supports both GoEmotions and ImgFlip datasets transparently
- **Baseline Results**: Comprehensive polarity analysis across emotion frequency categories

### Dataset Strategy Achievements

1. **Configurable Datasets**: Switch between `DATASET_NAME = "goemotions"` or `"imgflip"`
2. **Consistent Model Naming**: Dataset-specific paths (e.g., `gpt2-base-imgflip`)
3. **Balanced Training Data**: Applied hybrid sampling (downsample + upsample) to address class imbalance
4. **Complete Output Strategy**: No premature TOP/BOTTOM splitting, preserves emotional context

### Next Steps

The next notebook (**2b_caption_generation.ipynb**) will focus on:

- **Loading and using the fine-tuned ImgFlip models** for inference
- **Comprehensive evaluation metrics** (polarity analysis, semantic similarity, diversity analysis)
- **Comparison between baseline and fine-tuned performance** using balanced emotion categories
- **Production-ready caption generation functions** with intelligent post-processing
- **Integration-ready code** for the cartoonization pipeline using complete output strategy

### Key Innovation

Successfully adapted the unified dataset factory from 2a notebook to work with ImgFlip dataset, demonstrating the power of the **unified approach** while maintaining all the enhanced features and optimization from the GoEmotions implementation.